# GameTheory-13h — La coordination dans le temps, mesuree sur un jouet exact

> Sous-serie « autour de 13 » (information imparfaite). Compagnon de
> [GameTheory-13f-Ataraxos-Patron-Python.ipynb](GameTheory-13f-Ataraxos-Patron-Python.ipynb)
> (le patron en trois briques, grain A de #20408), de
> [GameTheory-13g-Ataraxos-Organe-Natif.ipynb](GameTheory-13g-Ataraxos-Organe-Natif.ipynb)
> (l'organe natif invoque, grain B) et de
> [GameTheory-13d-Optimistic-CFR-Python.ipynb](GameTheory-13d-Optimistic-CFR-Python.ipynb)
> (la famille de mise a jour dont les variantes sont mesurees ici).

**Objectifs** : mesurer le gap exact et comparer les quatre régimes de mise à jour.

**Prérequis** : stratégies comportementales, espérance et C#. **Durée** : 90 minutes.

**Navigation** : [Version Python](GameTheory-13h-Ataraxos-DynamicDamping-Python.ipynb) | [Index GameTheory](README.md) | [13d — Optimistic CFR](GameTheory-13d-Optimistic-CFR-Python.ipynb)

## La question

Ataraxos (arXiv 2511.07312) stabilise l'auto-jeu **sans** reweighting d'importance
des trajectoires ni moyennage de politiques. Sa these est une **coordination dans le
temps** entre deux leviers :

| Levier | Ce qu'il fait | Coordination proposee |
|---|---|---|
| force de **regularisation** (penalite vers la politique de collecte) | empeche la politique de s'eloigner des donnees qui l'ont produite | **forte tot**, faible tard |
| **taille du pas** de mise a jour | determine la vitesse d'adaptation | **agressive tot**, petite tard |

Cette these est une affirmation **mesurable** : si la coordination compte, un regime a
schemas variables doit battre un regime a parametres constants **a budget egal**.

## Ce que ce carnet mesure — et ce qu'il ne mesure pas

- **Mesure** : l'**exploitabilite exacte** (par meilleure reponse, sans echantillon,
  sans bruit) de quatre regimes de mise a jour, a six jalons de budget, sur le poker
  de Kuhn (3 cartes, equilibre connu : valeur `-1/18`).
- **Ne mesure pas** : Ataraxos lui-meme. Le papier coordonne pas et regularisation
  dans un auto-jeu **avec reseau** ; ici le jeu est tabulaire et la mise a jour est
  exacte. Le carnet mesure une **dynamique d'optimisation**, pas une force de jeu, et
  le transpose explicitement (section 6).

## Pourquoi un jouet a equilibre connu

Sur Kuhn, l'exploitabilite est calculable **exactement** : un joueur a 6 information
sets binaires, donc **64 strategies pures**, et la meilleure reponse s'obtient par
enumeration. Aucun echantillonnage, aucune graine : chaque chiffre du carnet se
**reproduit a l'identique** sur une meme plateforme.

Une réserve mesurée : à 2 000 itérations, le rejeu Python donne un gap `sched` de `0.03901`, et le jumeau C# `0.03895`. L'actualisation multiplicative amplifie les petits écarts flottants entre implémentations ; les deux classements de fin de budget coïncident. Les autres résultats affichés restent proches.

C'est ce qui rend l'ablation honnete. Sur un jeu a grande echelle, « le regime A semble
meilleur » peut etre un artefact de bruit d'estimation ; ici, la mesure est
l'exploitabilite vraie.

## Plan

1. Le jeu : arbre, payoffs, information sets.
2. L'exploitabilite exacte.
3. Le CFR comme reference de convergence.
4. Quatre regimes de mise a jour a budget egal.
5. Verdict mesure, avec son resultat negatif.
6. Limites de la transposition, exercices, bilan.

## 1. Le jeu : arbre, information sets, payoffs

Trois conventions portent tout le reste :

- **action 0 = check / coucher**, **action 1 = miser / suivre** — la numerotation est
  commune aux deux joueurs, mais le *sens* depend du contexte. C'est la fonction
  `child()` qui porte cette correspondance, et elle seule : confondre « action 1 » avec
  « miser » produit des historiques inexistants (`cbb`) et un arbre faux.
- **payoff du point de vue du joueur 0** (jeu a somme nulle : celui du joueur 1 est son
  oppose).
- **information set** = `(joueur, historique, carte)` : chaque joueur connait sa carte
  et l'historique, jamais la carte de l'autre.

Ante de 1 chacun. Un seul tour de mise. Pot de 2 apres deux checks, de 4 apres une
mise suivie.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Globalization;
CultureInfo.DefaultThreadCurrentCulture=CultureInfo.InvariantCulture;
Console.WriteLine($"Runtime : {System.Runtime.InteropServices.RuntimeInformation.FrameworkDescription}");
var deals=(from a in Enumerable.Range(0,3) from b in Enumerable.Range(0,3) where a!=b select (a,b)).ToArray();
record struct Node(int? Player,double Payoff);
Node Resolve(string h,int c0,int c1)=>h switch {""=>new Node(0,0),"c"=>new Node(1,0),"b"=>new Node(1,0),"cb"=>new Node(0,0),"bf"=>new Node(null,1),"cbf"=>new Node(null,-1),"cc"=>new Node(null,c0>c1?1:-1),"cbc" or "bc"=>new Node(null,c0>c1?2:-2),_=>throw new ArgumentException("Historique inconnu")};
string Child(string h,int a)=>h switch {"" or "c"=>h+(a==0?"c":"b"),"b" or "cb"=>h+(a==0?"f":"c"),_=>throw new ArgumentException("Pas de décision")};
int Key(int pl,string h,int card)=> (pl==0?(h==""?0:3):(h=="c"?0:3))+card;
Console.WriteLine($"Donnes : {deals.Length}, information sets par joueur : 6, stratégies pures : {1<<6}");

Runtime : .NET 10.0.14


Donnes : 6, information sets par joueur : 6, stratégies pures : 64


### Verification de l'arbre

Un arbre faux rendrait toute la mesure suivante sans valeur : on verifie donc cinq
payoffs a la main — les trois abattages et les deux couches — ainsi que la
correspondance action -> historique dans les deux contextes.

In [2]:
var checks=new[]{("cc",2,0,1.0),("cc",0,2,-1.0),("bc",2,0,2.0),("bf",0,2,1.0),("cbf",2,0,-1.0)};
foreach(var (h,c0,c1,want) in checks){double got=Resolve(h,c0,c1).Payoff;if(got!=want)throw new Exception("Paiement incorrect");Console.WriteLine($"{h} ({c0},{c1}) : {got:+0.0;-0.0} attendu {want:+0.0;-0.0}");}
if(Child("",0)!="c"||Child("",1)!="b"||Child("cb",0)!="cbf"||Child("cb",1)!="cbc")throw new Exception("Transition incorrecte");foreach(var h in new[]{"","c","b","cb"})foreach(int a in new[]{0,1})Resolve(Child(h,a),0,2);
Console.WriteLine("Arbre vérifié : 5 paiements et 8 transitions.");

cc (2,0) : +1.0 attendu +1.0


cc (0,2) : -1.0 attendu -1.0


bc (2,0) : +2.0 attendu +2.0


bf (0,2) : +1.0 attendu +1.0


cbf (2,0) : -1.0 attendu -1.0


Arbre vérifié : 5 paiements et 8 transitions.


## 2. L'exploitabilite exacte

L'exploitabilite d'un profil se mesure par **meilleure reponse** : combien un joueur
gagnerait-il en changeant entierement de strategie face a un adversaire fige ? Ici le
calcul est exact : **enumeration des 64 strategies pures** de chaque joueur.

Le profil est decrit par des **politiques comportementales** — une probabilite de jouer
l'action 1 (miser/suivre) par information set. L'instrument est le **gap de Nash** :

```
gap = ( valeur de la meilleure reponse de P0  +  valeur de la meilleure reponse de P1 ) / 2
```

A l'equilibre, la meilleure reponse de P0 vaut `v` et celle de P1 vaut `-v` : la somme
est nulle. Sur un profil quelconque, chacune est une valeur de meilleure réponse ; elles peuvent être de signes opposés. Leur moyenne donne le gain moyen d'une déviation unilatérale dans ce jeu à somme nulle.

In [3]:
double Ev(double[] p0,double[] p1){double Rec(string h,int c0,int c1){var node=Resolve(h,c0,c1);if(node.Player is null)return node.Payoff;int pl=node.Player.Value,card=pl==0?c0:c1;double p=(pl==0?p0:p1)[Key(pl,h,card)];return (1-p)*Rec(Child(h,0),c0,c1)+p*Rec(Child(h,1),c0,c1);}return deals.Average(d=>Rec("",d.a,d.b));}
double[] Pure(int index)=>Enumerable.Range(0,6).Select(i=>(double)((index>>i)&1)).ToArray();
double BestResponse(double[] opp,int player)=>Enumerable.Range(0,64).Max(k=>player==0?Ev(Pure(k),opp):-Ev(opp,Pure(k)));
double Gap(double[] p0,double[] p1)=>(BestResponse(p1,0)+BestResponse(p0,1))/2;
var zero=Enumerable.Repeat(0.0,6).ToArray();var one=Enumerable.Repeat(1.0,6).ToArray();var half=Enumerable.Repeat(.5,6).ToArray();
foreach(var (nom,p) in new[]{("tout coucher/check",zero),("tout miser/suivre",one),("uniforme",half)})Console.WriteLine($"{nom,-20} valeur={Ev(p,p):F4}, gap={Gap(p,p):F4}");

tout coucher/check   valeur=0.0000, gap=1.0000


tout miser/suivre    valeur=0.0000, gap=0.3333


uniforme             valeur=0.1250, gap=0.4583


Les trois profils degeneres sont fortement exploitables : l'instrument distingue bien
« mauvais » de « bon ». Il reste a verifier qu'il vaut **zero** la ou personne ne peut
rien reprocher a personne. C'est le role de la reference CFR : un algorithme dont la
convergence est connue, mesure par le meme instrument, doit rendre un gap qui tend vers
zero et une valeur qui tend vers `-1/18`.

## 3. Le CFR comme reference de convergence

Le **counterfactual regret minimisation** est l'algorithme de reference sur les jeux a
information imparfaite : regrets contrefactuels accumules par information set,
politique obtenue par *regret matching*, et moyenne temporelle des politiques.

Il sert ici de **temoin d'instrument**, pas de sujet : s'il ne converge pas vers `-1/18`
avec un gap qui decroit, c'est l'instrument qu'il faut corriger.

> La recursion renvoie l'utilite du joueur courant et accumule les regrets ponderes par
> la portee adverse — c'est la definition contrefactuelle, et c'est ce qui fait la
> difference avec un simple gradient de politique.

In [4]:
double[] RegretMatch(double[] r){double a=Math.Max(r[0],0),b=Math.Max(r[1],0),s=a+b;return s>0?new[]{a/s,b/s}:new[]{.5,.5};}
double[] Softmax(double[] z){double m=z.Max();var e=z.Select(x=>Math.Exp(x-m)).ToArray();double sum=e.Sum();return e.Select(x=>x/sum).ToArray();}
record Mesure(double Gap,double Valeur,double[] P0,double[] P1);
Dictionary<int,Mesure> Train(string regime,int iterations,int[] jalons,int horizon=2000,Func<double[],double[]> softRule=null){
 var regret=Enumerable.Range(0,2).Select(_=>Enumerable.Range(0,6).Select(_=>new double[2]).ToArray()).ToArray();
 var somme=Enumerable.Range(0,2).Select(_=>Enumerable.Range(0,6).Select(_=>new double[2]).ToArray()).ToArray();var seen=new Dictionary<int,Mesure>();
 double Walk(string h,int c0,int c1,double reach0,double reach1,int player,int t){var node=Resolve(h,c0,c1);if(node.Player is null)return player==0?node.Payoff:-node.Payoff;int pl=node.Player.Value,card=pl==0?c0:c1,key=Key(pl,h,card);var sigma=RegretMatch(regret[pl][key]);double[] vals=new double[2];for(int a=0;a<2;a++)vals[a]=pl==0?Walk(Child(h,a),c0,c1,reach0*sigma[a],reach1,player,t):Walk(Child(h,a),c0,c1,reach0,reach1*sigma[a],player,t);double moyenne=sigma[0]*vals[0]+sigma[1]*vals[1];if(pl==player){double own=pl==0?reach0:reach1,opp=pl==0?reach1:reach0;double weight=regime=="linavg"?t:1;for(int a=0;a<2;a++){regret[pl][key][a]+=opp*(vals[a]-moyenne);somme[pl][key][a]+=weight*own*sigma[a];}}return moyenne;}
 for(int t=1;t<=iterations;t++){foreach(var (c0,c1) in deals)for(int player=0;player<2;player++)Walk("",c0,c1,1,1,player,t);
  if(regime=="cfrplus"){for(int pl=0;pl<2;pl++)for(int k=0;k<6;k++)for(int a=0;a<2;a++)regret[pl][k][a]=Math.Max(regret[pl][k][a],0);}
  else if(regime=="sched"){double aa=3*Math.Pow(.1/3,t/(double)horizon),gg=2.5*Math.Pow(1/2.5,t/(double)horizon),pow=Math.Pow(t,aa),posW=pow/(pow+1),avgW=Math.Pow(t/(t+1.0),gg);for(int pl=0;pl<2;pl++)for(int k=0;k<6;k++)for(int a=0;a<2;a++){if(regret[pl][k][a]>0)regret[pl][k][a]*=posW;somme[pl][k][a]*=avgW;}}
  else if(regime=="soft"){if(softRule==null)throw new ArgumentException("Règle soft manquante");for(int pl=0;pl<2;pl++)for(int k=0;k<6;k++)regret[pl][k]=softRule((double[])regret[pl][k].Clone());}
  if(jalons.Contains(t)){var avg=Enumerable.Range(0,2).Select(pl=>Enumerable.Range(0,6).Select(k=>{double a=somme[pl][k][0],b=somme[pl][k][1];return a+b>0?b/(a+b):.5;}).ToArray()).ToArray();seen[t]=new Mesure(Gap(avg[0],avg[1]),Ev(avg[0],avg[1]),avg[0],avg[1]);}}
 return seen;
}
var reference=Train("plain",20000,new[]{100,1000,5000,20000});foreach(var (t,m) in reference)Console.WriteLine($"{t,6} gap={m.Gap:F5} valeur={m.Valeur:F5}");Console.WriteLine($"Valeur théorique : {-1.0/18:F5}");

   100 gap=0.01685 valeur=-0.05320


  1000 gap=0.00604 valeur=-0.05549


  5000 gap=0.00270 valeur=-0.05556


 20000 gap=0.00128 valeur=-0.05556


Valeur théorique : -0.05556


Le témoin fournit un contrôle de cohérence de l'instrument : le gap atteint le millième et la valeur approche `-1/18`. Les valeurs suivantes restent des mesures exactes pour les profils calculés, sous réserve de l'implémentation et des arrondis.

## 4. Quatre regimes de mise a jour, a budget egal

Pour étudier une analogie avec la coordination temporelle d'Ataraxos, ce jouet CFR combine une correction forte tôt et faible tard avec une pondération de la moyenne temporelle. Ces mécanismes sont propres au jouet ; le papier ne moyenne pas les politiques. On compare ici :

| Regime | Correction (regrets positifs) | Moyenne | Ce qu'il isole |
|---|---|---|---|
| `plain` (temoin negatif) | aucune actualisation | uniforme | ni coordination ni ponderation |
| `cfrplus` | regrets negatifs remis a zero a chaque passe | uniforme | agressivite **constante** |
| `linavg` | aucune actualisation | poids `t` | ponderation des recentes, **constante** |
| `sched` | facteur `t^a/(t^a+1)`, `a` de 3 vers 0.1 | poids `(t/(t+1))^g`, `g` de 2.5 vers 1 | **coordination planifiee** des deux leviers |

`sched` est le candidat « coordination » : les deux leviers sont agressifs au debut et
doux a la fin. S'il gagne a budget egal sur `plain`, la coordination explique quelque
chose ; sinon, elle n'explique rien ici et il faut le dire.

In [5]:
var checkpoints=new[]{25,50,100,200,400,2000};var regimes=new[]{"plain","cfrplus","linavg","sched"};
var results=regimes.ToDictionary(r=>r,r=>Train(r,2000,checkpoints));
foreach(var r in regimes)Console.WriteLine($"{r,-9} "+string.Join(" ",checkpoints.Select(t=>results[r][t].Gap.ToString("F5"))));

plain     0.04494 0.02824 0.01685 0.01225 0.00914 0.00431


cfrplus   0.04479 0.03163 0.02221 0.01552 0.01086 0.00484


linavg    0.05275 0.02879 0.01218 0.00900 0.00670 0.00303


sched     0.01755 0.01037 0.01581 0.01205 0.00647 0.03895


### Exercice 1 — Atténuation constante

Complétez `regleSoft` : multipliez les regrets positifs par 0,9 et conservez les regrets nuls ou négatifs. Le paramètre `softRule` de `Train("soft",...)` utilise votre fonction à chaque passage ; comparez les gaps à 200 et 2 000 itérations.

In [6]:
Func<double[],double[]> regleSoft=regrets=>{
 // TODO : atténuer les seuls regrets positifs
 return null;
};
var sondesSoft=new[]{new[]{-2.0,3.0},new[]{0.0,1.0},new[]{4.0,-1.0}};var attenduSoft=new[]{new[]{-2.0,2.7},new[]{0.0,.9},new[]{3.6,-1.0}};var repSoft=sondesSoft.Select(x=>regleSoft((double[])x.Clone())).ToArray();
if(repSoft.Any(x=>x==null))Console.WriteLine("Exercice 1 à compléter");
else{for(int i=0;i<repSoft.Length;i++)if(repSoft[i].Length!=2||repSoft[i].Any(x=>!double.IsFinite(x))||Enumerable.Range(0,2).Any(j=>Math.Abs(repSoft[i][j]-attenduSoft[i][j])>1e-12))throw new Exception("Sonde incorrecte");var mesureSoft=Train("soft",2000,checkpoints,softRule:regleSoft);Console.WriteLine($"Exercice 1 : gaps soft à 200 et 2000 : {mesureSoft[200].Gap:F5}, {mesureSoft[2000].Gap:F5}");}

Exercice 1 à compléter


## 5. Verdict mesure

Trois lectures, dans cet ordre : la convergence finale (les quatre regimes atteignent-ils
`-1/18` ?), le classement a budget court, et le signe de l'ecart au dernier jalon. La
cellule suivante les imprime ; les nombres du verdict sont ceux de la sortie, pas d'une
estimation.

In [7]:
foreach(var r in regimes){var vals=results[r];int best=vals.OrderBy(kv=>kv.Value.Gap).First().Key;Console.WriteLine($"{r,-9} gap final={vals[2000].Gap:F5}, valeur={vals[2000].Valeur:F5}, meilleur jalon={best}");}
foreach(int t in new[]{200,2000}){double baseline=results["plain"][t].Gap;foreach(var r in regimes)Console.WriteLine($"à {t} : {r,-9} gap={results[r][t].Gap:F5}, écart vs plain={100*(results[r][t].Gap/baseline-1):F1} %");}

plain     gap final=0.00431, valeur=-0.05555, meilleur jalon=2000


cfrplus   gap final=0.00484, valeur=-0.05592, meilleur jalon=2000


linavg    gap final=0.00303, valeur=-0.05555, meilleur jalon=2000


sched     gap final=0.03895, valeur=-0.05331, meilleur jalon=400


à 200 : plain     gap=0.01225, écart vs plain=0.0 %


à 200 : cfrplus   gap=0.01552, écart vs plain=26.7 %


... (2 outputs left out)

à 2000 : plain     gap=0.00431, écart vs plain=0.0 %


à 2000 : cfrplus   gap=0.00484, écart vs plain=12.3 %


à 2000 : linavg    gap=0.00303, écart vs plain=-29.7 %


à 2000 : sched     gap=0.03895, écart vs plain=803.2 %


### Lecture

Trois faits, que la sortie ci-dessus etablit :

1. **Trois regimes sur quatre approchent à ce budget la valeur de référence** `-1/18` : `plain` (`-0.05555`, ecart
   `6e-6`), `linavg` (`-0.05555`, ecart `6e-6`) et `cfrplus` (`-0.05592`, ecart `3,6e-4`).
   **`sched` fait exception** : `-0.05331`, soit un ecart de `2,2e-3`, six fois celui de
   `cfrplus` ; et son meilleur jalon est 400, pas 2000. A ce budget il n'a donc **pas**
   converge : sa valeur finale est une valeur de **budget fini**, pas un point fixe
   atteint. Ce n'est pas un defaut d'instrument — l'ecart est mesure exactement — c'est
   le resultat, et le point 3 le reprend.
2. **La ponderation des iterations recentes (`linavg`) gagne a budget court** et garde
   son avantage a 2000 iterations : c'est l'ingredient « agressif tot » qui se transpose
   le mieux sur ce jouet.
3. **Le regime a coordination planifiee (`sched`) mene tres tot** (0.01755 a 25
   iterations, contre 0.04494 pour `plain`), puis **perd** en fin de budget : son gap
   descend jusqu'a 0.00647 a 400 iterations, puis **remonte** a 0.03895 a 2000. Ses schemas
   sont calibres sur un horizon de 2000 iterations, et l'actualisation des regrets est
   multiplicative : elle modifie l'accumulation du CFR ; cette trajectoire finie ne démontre pas à elle seule l'absence de convergence asymptotique. C'est un
   **resultat negatif mesure**, et il est instructif : la coordination dans le temps n'est
   pas gratuite — elle ne permet pas ici d'invoquer sans preuve la garantie du CFR standard et montre un compromis à budget fini.

`cfrplus` (agressivite **constante**) est le moins performant des trois regimes non planifies au dernier jalon : la remise à zéro systématique des regrets négatifs n'est pas la plus efficace dans cette expérience à budget fini. Autrement
dit, ce qui aide est la **ponderation temporelle**, pas l'agressivite seule.



### Exercice 2 — Horizon des schémas

Renseignez `comparaisonHorizons` avec les résultats de `Train("sched",2000,checkpoints,horizon)` pour les horizons 200, 400 et 2 000. Comparez les gaps aux six jalons ; le garde recalcule les mesures.

In [8]:
Dictionary<int,Dictionary<int,Mesure>> comparaisonHorizons=null; // TODO : exécuter sched aux horizons 200, 400, 2000
if(comparaisonHorizons==null)Console.WriteLine("Exercice 2 à compléter");
else{if(!comparaisonHorizons.Keys.OrderBy(x=>x).SequenceEqual(new[]{200,400,2000}))throw new Exception("Horizons incorrects");foreach(int horizon in new[]{200,400,2000}){var attendu=Train("sched",2000,checkpoints,horizon);var d=comparaisonHorizons[horizon];if(d==null||!d.Keys.OrderBy(x=>x).SequenceEqual(checkpoints))throw new Exception("Jalons incorrects");foreach(int t in checkpoints){var m=d[t];if(m==null||!double.IsFinite(m.Gap)||!double.IsFinite(m.Valeur)||Math.Abs(m.Gap-attendu[t].Gap)>1e-10||Math.Abs(m.Valeur-attendu[t].Valeur)>1e-10)throw new Exception("Mesure incorrecte");}}Console.WriteLine("Exercice 2 : horizons et jalons vérifiés");}

Exercice 2 à compléter


## 6. Limites de la transposition

Ce carnet ne reproduit pas Ataraxos, et ne le pretend pas. Quatre ecarts, a lire avant de
citer un chiffre :

1. **Tabulaire contre reseau.** Ici la politique est une table de 24 probabilites et les
   valeurs contrefactuelles sont **exactes**. Ataraxos optimise un reseau par descente de
   gradient estimee : la dynamique mesuree ici est celle de l'optimisation, pas celle de
   l'approximation.
2. **Budget.** 2000 iterations d'un jeu a 3 cartes. Un avantage a budget court ne dit rien
   de la vitesse asymptotique.
3. **La regularisation n'est pas la meme.** Le `sched` transpose la penalite reverse-KL
   d'Ataraxos par une actualisation des regrets : deux mecanismes voisins, pas identiques.
   Le carnet mesure la these generale « coordonner les leviers dans le temps », pas
   l'implementation du papier.
4. **Un seul jeu.** Kuhn est le plus petit jeu a information imparfaite interessant ; c'est
   aussi un jeu ou les strategies mixtes sont peu nombreuses, ce qui favorise les methodes
   d'accumulation.

Le claim « 1/500 du calcul » d'Ataraxos n'est **pas** mesurable ici et n'est pas revendique.

## 7. Exercices — récapitulatif

Les trois exercices répartis ci-dessus interrogent une atténuation constante, l'horizon du schéma et les valeurs séparées de meilleure réponse. Ils utilisent les mêmes budgets et les mêmes jalons que la comparaison principale.

### Exercice 3 — Valeurs séparées de meilleure réponse

Complétez `valeursSeparees` pour retourner les valeurs de meilleure réponse des deux joueurs. Le garde teste plusieurs profils, puis mesure `plain` aux jalons 200 et 2 000.

In [9]:
Func<double[],double[],(double,double)?> valeursSeparees=(p0,p1)=>{
 // TODO : valeurs de meilleure réponse de P0 et P1
 return null;
};
var profils=new[]{(zero,zero),(one,one),(half,half)};var repBr=profils.Select(x=>valeursSeparees(x.Item1,x.Item2)).ToArray();
if(repBr.Any(x=>x is null))Console.WriteLine("Exercice 3 à compléter");
else{for(int i=0;i<repBr.Length;i++){var x=repBr[i].Value;var p=profils[i];if(!double.IsFinite(x.Item1)||!double.IsFinite(x.Item2)||Math.Abs(x.Item1-BestResponse(p.Item2,0))>1e-10||Math.Abs(x.Item2-BestResponse(p.Item1,1))>1e-10)throw new Exception("Meilleure réponse incorrecte");}var plain=Train("plain",2000,checkpoints);foreach(int t in new[]{200,2000}){var m=plain[t];var x=valeursSeparees(m.P0,m.P1);if(x is null||Math.Abs((x.Value.Item1+x.Value.Item2)/2-m.Gap)>1e-10)throw new Exception("Gap incohérent");Console.WriteLine($"Exercice 3 : jalon {t}, valeurs BR={x}");}}

Exercice 3 à compléter


## À retenir

### Trois mesures à retenir

1. **Le jouet a une valeur et un gap exacts.** La valeur du jeu est **−1/18 = −0,05556**, et les profils dégénérés donnent les repères : tout coucher / tout suivre → gap **1,0000**, tout miser / tout suivre → gap **0,3333**, stratégie uniforme → gap **0,4583**. L'exploitabilité est calculée par meilleure réponse sur les **64** stratégies pures, sans échantillonnage.
2. **L'instrument converge, mais lentement.** Le CFR de référence descend de **0,01685** (100 itérations) à **0,00604** (1 000), **0,00270** (5 000) et **0,00128** (20 000), vers la valeur **−0,05556** : le gap décroît d'un facteur 13 entre 100 et 20 000 itérations.
3. **La coordination planifiée est un résultat négatif.** À budget égal (2 000 itérations), le régime `sched` finit à **0,03895** contre **0,00431** pour le régime nu — soit **+803,2 %** — et son meilleur jalon est **400** itérations (**0,00647**) : le calendrier actualisé est utile très tôt mais se dégrade nettement au dernier jalon sur ce jeu.

| Element | Etat |
|---|---|
| Arbre de Kuhn (payoffs, information sets, transitions) | implemente, verifie par 5 asserts de payoff et 8 transitions |
| Exploitabilite exacte par meilleure reponse | implementee (64 strategies pures par joueur, aucun echantillonnage) |
| Instrument valide par le CFR | gap decroissant vers le millieme, valeur `-1/18` |
| Quatre regimes compares a budget egal | mesures a 6 jalons |
| Resultat negatif (coordination planifiee) | mesure et interprete |
| Limites de la transposition | section 6 |

**Ce que le carnet ajoute** : une ablation **exacte et rejouable** de la these de
coordination temporelle, avec son temoin negatif, dans la sous-serie 13. Chaque chiffre se
reproduit a l'identique d'une execution a l'autre, sauf le dernier jalon de `sched`, dont
le quatrieme decimal depend de la plateforme (ulp de `pow()` amplifiees par l'actualisation
multiplicative ; voir « Pourquoi un jouet a equilibre connu »). Le carnet ne reproduit pas
Ataraxos et ne le pretend pas.


## Vérifiez votre compréhension

1. Pourquoi le carnet utilise-t-il un jeu dont l'équilibre est connu ?
2. Comment se comporte l'exploitabilité du CFR de référence au fil des itérations ?
3. Le régime `sched` est-il le meilleur à 2 000 itérations ? Que disent les chiffres ?
4. À 200 itérations, le classement des quatre régimes est-il le même qu'à 2 000 ?

<details><summary>Réponses</summary>

1. Pour que l'aptitude à jouer soit vérifiable sans solveur de poker : la valeur du jeu (**−1/18**) et l'exploitabilité exacte sont calculables par énumération sur les 64 stratégies pures. Chaque régime est alors jugé par un nombre, pas par une impression.
2. Le gap décroît aux jalons — **0,01685** (100 itérations), **0,00604** (1 000), **0,00270** (5 000), **0,00128** (20 000) — tandis que la valeur estimée se rapproche de **−0,05556**. Le CFR sert de référence, pas de champion : il lui faut un facteur 13 entre 100 et 20 000 itérations.
3. Non, c'est l'inverse : `sched` termine à **0,03895** contre **0,00431** pour le régime nu, soit **+803,2 %**, et son meilleur jalon est **400** itérations (**0,00647**) — le calendrier actualisé dégrade la mise à jour au lieu de l'améliorer.
4. Non : à 200 itérations, `cfrplus` est **+26,7 %** et `linavg` **−26,5 %** par rapport à `plain` ; à 2 000, `cfrplus` tombe à **+12,3 %** et `linavg` reste à **−29,7 %**, tandis que `sched` passe de **−1,6 %** à **+803,2 %**. Le classement change avec le budget.

</details>


## Pour aller plus loin

**Lectures.** — les sources primaires des quatre régimes de mise à jour :

- Russell & Norvig, *AIMA* (4e ed.), ch. 17 — jeux a information imparfaite, equilibre de
  Nash, regret.
- Kuhn, H. W. (1950). *Simplified Poker*. Contributions to the Theory of Games I. (valeur
  `-1/18` du poker a trois cartes)
- Zinkevich, M., Johanson, M., Bowling, M., Piccione, C. (2007). *Regret Minimization in
  Games with Incomplete Information* (NeurIPS) — CFR.
- Tammelin, O., Burch, N., Johanson, M., Bowling, M. (2015). *Solving Heads-up Limit Texas
  Hold'em* (IJCAI) — CFR+ et la question de l'agressivite constante.
- Brown, N., Sandholm, T. (2019). *Solving Imperfect-Information Games via Discounted
  Regret Minimization* (AAAI) — actualisation planifiee, transposee ici en regime `sched`.
- Sokota, S., Vinitsky, E., Hu, H., Fan, Z., Kolter, J. Z., Farina, G. (2025/2026).
  *Scalable Decision Making for Games of Imperfect Information* — arXiv
  [2511.07312](https://arxiv.org/abs/2511.07312).
- [GameTheory-13d-Optimistic-CFR-Python.ipynb](GameTheory-13d-Optimistic-CFR-Python.ipynb)
  — la famille de mise a jour dont les variantes sont mesurees ici.
- [GameTheory-13f-Ataraxos-Patron-Python.ipynb](GameTheory-13f-Ataraxos-Patron-Python.ipynb)
  — le patron en trois briques (grain A de #20408).
- [GameTheory-13g-Ataraxos-Organe-Natif.ipynb](GameTheory-13g-Ataraxos-Organe-Natif.ipynb)
  — l'organe natif `AtaraxosAI/doudizhu` invoque (grain B de #20408).

***


**Carnet suivant.** : [GameTheory-14 — Jeux différentiels et équilibres de Stackelberg](GameTheory-14-DifferentialGames-Python.ipynb) — l'information imparfaite et l'équilibre de Stackelberg.

**Dans la même série.** : [GameTheory-13d — Optimistic CFR](GameTheory-13d-Optimistic-CFR-Python.ipynb) — la famille de mise à jour dont les variantes sont mesurées ici ; [GameTheory-13f — Ataraxos, le patron](GameTheory-13f-Ataraxos-Patron-Python.ipynb) — le patron en trois briques ; [GameTheory-13g — Ataraxos, l'organe natif](GameTheory-13g-Ataraxos-Organe-Natif.ipynb) — le même patron sur l'organe réel.


***

**Navigation** : [<< 13g-Ataraxos-Organe-Natif](GameTheory-13g-Ataraxos-Organe-Natif.ipynb) | [Index](README.md)
